# Week06 Part 2 — RAG 기반 질문 카테고리 분류 (AI API) [완성본]

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itlects/2026-1_ai-convergence-apply/blob/main/Week06/Part2/Week06_Part2_RAG_Classification_%EC%99%84%EC%84%B1%EB%B3%B8.ipynb)

**목표:** 카테고리별 예시 질문을 지식베이스로 두고, 새 질문과 가장 비슷한 예시를 검색(Retrieval)해 질문을 사전 정의된 카테고리로 분류한다. 유사도 수치로 근거를 확인하고, 검색 결과를 생성형 AI에 넣어 분류 이유까지 설명하게 한다.

**학습 방법:** 모든 코드가 완성된 버전입니다. 위에서부터 순서대로 실행하면 결과를 확인할 수 있습니다.

## 실습 0. API 키 설정 (Colab Secrets)
1. Colab 왼쪽 **🔑(Secrets)** 아이콘 → **새 보안 비밀 추가**
2. 이름 `OPENAI_API_KEY`, 값에 본인 키 입력 → **노트북 액세스** 켜기
3. 키는 **절대** 코드 셀, 스크린샷, GitHub, 과제 제출물에 넣지 않습니다.

> 키가 없어도 됩니다. 키가 없으면 Part 1의 무료 한국어 임베딩 모델(강의자 Google Drive에서 복사)로 자동 전환되어 실습 1~5를 모두 할 수 있고, 실습 6(생성형 AI 분류)만 건너뜁니다.
> API 호출은 소액 과금될 수 있습니다(임베딩은 매우 저렴, 이 노트북 전체 실행 시 보통 수십 원 이하).

In [ ]:
import importlib.util
if importlib.util.find_spec("openai") is None:
    !pip install -q openai

import os, json
import numpy as np
import pandas as pd

try:
    from google.colab import userdata
    API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    API_KEY = os.environ.get("OPENAI_API_KEY")

EMBED_MODEL = "text-embedding-3-small"   # OpenAI 임베딩 모델
CHAT_MODEL = "gpt-4o-mini"               # 실습 6에서 사용할 생성 모델 (수업 시점에 사용 가능한 모델로 변경 가능)

if API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY)
    MODE = "openai"
else:
    client = None
    MODE = "local"
print("API 키 설정 여부:", bool(API_KEY), "/ 실행 모드:", MODE)

# ── 한국어 임베딩 모델 준비: 인터넷 다운로드 대신 강의자 Google Drive에서 복사 ──
import os, shutil, time

MODEL_FOLDER_ID = "1EmofMTJ6HRQrth8puRpHwvFDJ27YeiVr"   # 강의자 Drive의 모델 폴더(링크 공유)
SHORTCUT_DIR = "/content/drive/MyDrive/AI융합활용_6주_모델/ko-sroberta-multitask"  # 방법 2용 바로가기 경로
LOCAL_MODEL_DIR = "/content/models/ko-sroberta-multitask"

def model_ready(path):
    return (os.path.isdir(path) and os.path.exists(os.path.join(path, "modules.json"))
            and any(f.endswith((".safetensors", ".bin")) for f in os.listdir(path)))

def prepare_model():
    if model_ready(LOCAL_MODEL_DIR):
        return LOCAL_MODEL_DIR
    # 방법 1: 공유 링크로 강의자 Drive에서 바로 복사 (별도 설정 없음)
    try:
        import gdown
        print("방법 1: 강의자 Google Drive에서 모델을 복사합니다...")
        gdown.download_folder(id=MODEL_FOLDER_ID, output=LOCAL_MODEL_DIR, quiet=True)
        if model_ready(LOCAL_MODEL_DIR):
            return LOCAL_MODEL_DIR
    except Exception as e:
        print("  링크 복사 실패:", type(e).__name__)
    # 방법 2: 내 드라이브에 추가해 둔 바로가기에서 복사
    try:
        from google.colab import drive
        print("방법 2: 내 드라이브의 바로가기에서 복사합니다. (드라이브 연결을 허용하세요)")
        drive.mount("/content/drive")
        if model_ready(SHORTCUT_DIR):
            shutil.copytree(SHORTCUT_DIR, LOCAL_MODEL_DIR, dirs_exist_ok=True)
            return LOCAL_MODEL_DIR
        print("  바로가기를 찾지 못했습니다:", SHORTCUT_DIR)
    except Exception as e:
        print("  드라이브 복사 실패:", type(e).__name__)
    # 방법 3: 인터넷에서 직접 다운로드 (느림, 최후의 수단)
    print("방법 3: 인터넷에서 직접 다운로드합니다. 시간이 오래 걸릴 수 있습니다.")
    return "jhgan/ko-sroberta-multitask"


## 실습 1. 카테고리 지식베이스 만들기
'학과 도우미 챗봇'이 받을 질문을 4개 카테고리로 나눕니다. 카테고리마다 **예시 질문**을 모아 둔 것이 검색 대상(지식베이스)입니다.

In [ ]:
knowledge = {
    "수강·학사": [
        "수강신청 기간이 언제인가요?",
        "수강 정정은 어떻게 하나요?",
        "휴학 신청 절차를 알려주세요",
        "졸업 학점이 몇 학점이에요?",
        "성적 이의신청은 언제까지 하나요?",
        "재수강하면 성적이 어떻게 반영되나요?",
    ],
    "장학·등록금": [
        "국가장학금 신청 기간이 궁금해요",
        "성적장학금 기준이 뭔가요?",
        "등록금 분할 납부가 가능한가요?",
        "장학금은 언제 입금되나요?",
        "근로장학생은 어떻게 지원하나요?",
        "등록금 납부 영수증은 어디서 출력해요?",
    ],
    "실습실·장비": [
        "실습실 개방 시간이 어떻게 되나요?",
        "오실로스코프 사용법을 알려주세요",
        "실습실 컴퓨터가 고장 났어요",
        "3D 프린터 예약은 어떻게 하나요?",
        "납땜 장비를 빌릴 수 있나요?",
        "주말에도 실습실을 쓸 수 있나요?",
    ],
    "취업·자격증": [
        "전자기능사 시험 일정이 궁금해요",
        "취업 상담은 어디서 받나요?",
        "현장실습 신청 방법을 알려주세요",
        "이력서 첨삭을 받을 수 있나요?",
        "정보처리기능사 특강이 있나요?",
        "채용 공고는 어디서 확인하나요?",
    ],
}

kb = [{"category": cat, "text": q} for cat, qs in knowledge.items() for q in qs]
print("카테고리:", list(knowledge))
print("예시 질문 수:", len(kb))

## 실습 2. 임베딩 함수 만들고 지식베이스 임베딩하기
`embed()`는 문장 목록을 받아 **길이 1로 정규화된 벡터**를 돌려줍니다. 정규화된 벡터끼리는 내적(`@`)이 곧 코사인 유사도입니다.
- 키가 있으면: OpenAI Embeddings API 호출 (요청 → 응답의 `data[i].embedding`)
- 키가 없으면: Part 1의 무료 한국어 모델 사용 (인터넷 다운로드 없이 강의자 Drive에서 복사)

In [ ]:
if MODE == "local":
    if importlib.util.find_spec("sentence_transformers") is None:
        !pip install -q sentence-transformers
    from sentence_transformers import SentenceTransformer
    local_model = SentenceTransformer(prepare_model())   # 실습 0에서 정의한 함수: 강의자 Drive에서 복사

def embed(texts):
    if MODE == "openai":
        response = client.embeddings.create(model=EMBED_MODEL, input=texts)
        vecs = np.array([item.embedding for item in response.data])
    else:
        vecs = np.array(local_model.encode(texts))
    return vecs / np.linalg.norm(vecs, axis=1, keepdims=True)

kb_vecs = embed([item["text"] for item in kb])
print("지식베이스 벡터 모양:", kb_vecs.shape)

**확인:** OpenAI 모드와 무료 모드의 벡터 차원은 각각 얼마인가요? (1536 / 768)

## 실습 3. 검색 기반 질문 분류 (RAG 분류)
1. 새 질문을 임베딩한다.
2. 지식베이스에서 가장 비슷한 예시 질문 `k`개를 **검색**한다.
3. 검색된 예시들의 카테고리별 **유사도 합**이 가장 큰 카테고리로 분류한다.
4. 1순위 유사도가 `threshold`보다 낮으면 **"분류 불가"**로 처리한다.

In [ ]:
def classify(question, k=3, threshold=0.4, show=True):
    q = embed([question])[0]
    scores = kb_vecs @ q
    top = scores.argsort()[::-1][:k]

    votes = {}
    for i in top:
        votes[kb[i]["category"]] = votes.get(kb[i]["category"], 0) + float(scores[i])
    best = max(votes, key=votes.get)
    if scores[top[0]] < threshold:
        best = "분류 불가"

    if show:
        print(f"질문: {question}")
        for rank, i in enumerate(top, 1):
            print(f"  {rank}위 {scores[i]:.3f}  [{kb[i]['category']}] {kb[i]['text']}")
        print(f"  → 예측 카테고리: {best}\n")
    return best, [(kb[i]["category"], kb[i]["text"], round(float(scores[i]), 3)) for i in top]

classify("학비를 나눠서 내도 되나요?")
classify("랩실 PC 전원이 안 켜져요")
classify("오늘 학식 메뉴 뭐야?");

**확인:** '학비를 나눠서'는 지식베이스의 '등록금 분할 납부'와 단어가 다른데도 올바르게 분류되나요? '학식 메뉴'는 1순위 점수가 얼마인가요?
> 임계값은 임베딩 모델마다 적절한 값이 다릅니다. 무료 모델과 OpenAI 모델의 점수 범위를 비교해 보고 필요하면 `threshold`를 조정하세요.

## 실습 4. 분류 성능 평가
정답이 있는 테스트 질문으로 **정확도**와 **혼동표**(어떤 카테고리를 어떤 카테고리로 헷갈렸는지)를 확인합니다.

In [ ]:
test_set = [
    ("이번 학기 수강 철회 기간 알려줘", "수강·학사"),
    ("복학하려면 뭘 제출해야 해?", "수강·학사"),
    ("학점이 부족하면 졸업 못 하나요?", "수강·학사"),
    ("등록금 고지서는 어디서 봐요?", "장학·등록금"),
    ("장학금 받으려면 평점이 몇 이상이어야 해?", "장학·등록금"),
    ("교내 근로 자리 아직 있나요?", "장학·등록금"),
    ("멀티미터 대여 가능한가요?", "실습실·장비"),
    ("실습실 저녁에도 열려 있어요?", "실습실·장비"),
    ("인두기가 고장 났는데 어디에 말해요?", "실습실·장비"),
    ("자격증 시험 접수는 언제예요?", "취업·자격증"),
    ("면접 준비 도와주는 프로그램 있어요?", "취업·자격증"),
    ("기업 현장실습 학점 인정되나요?", "취업·자격증"),
    ("오늘 날씨 어때?", "분류 불가"),
    ("좋아하는 노래 추천해줘", "분류 불가"),
]

results = []
for question, answer in test_set:
    pred, hits = classify(question, show=False)
    results.append({"질문": question, "정답": answer, "예측": pred,
                    "1순위 유사도": hits[0][2], "정답 여부": pred == answer})

df = pd.DataFrame(results)
print(f"정확도: {df['정답 여부'].mean():.0%}  ({df['정답 여부'].sum()}/{len(df)})")
display(df)
display(pd.crosstab(df["정답"], df["예측"]))

**확인:** 틀린 질문은 무엇이고 왜 틀렸을까요? 지식베이스에 예시 질문을 보강하면(실습 1) 정확도가 올라가는지 다시 실행해 보세요.

## 실습 5. 임계값(threshold) 바꿔 보기
임계값이 너무 낮으면 엉뚱한 질문도 억지로 분류하고, 너무 높으면 정상 질문까지 '분류 불가'가 됩니다.

In [ ]:
rows = []
for th in [0.2, 0.3, 0.4, 0.5, 0.6]:
    correct = sum(classify(q, threshold=th, show=False)[0] == a for q, a in test_set)
    rows.append({"threshold": th, "정확도": f"{correct / len(test_set):.0%}"})
pd.DataFrame(rows)

**확인:** 정확도가 가장 높은 임계값은 얼마인가요? 이 값을 결과표에 기록하세요. (12주차 '인자 조율'에서 같은 개념을 다시 다룹니다.)

## 실습 6. 생성형 AI로 분류하고 이유 설명하기 (API 키 필요)
RAG의 핵심 = **검색한 근거를 프롬프트에 넣어 생성 모델이 답하게 하는 것**.
실습 3에서 검색된 예시 질문을 프롬프트에 넣고, LLM이 카테고리와 **분류 이유**를 JSON으로 답하게 합니다.

In [ ]:
CATEGORIES = list(knowledge) + ["분류 불가"]

def llm_classify(question, k=5):
    _, hits = classify(question, k=k, show=False)
    examples = "\n".join(f"- [{cat}] {text} (유사도 {score})" for cat, text, score in hits)
    prompt = (
        "너는 전자공학과 도우미 챗봇의 질문 분류기다.\n"
        f"카테고리 목록: {CATEGORIES}\n"
        "아래 '검색된 예시'만 근거로 사용자 질문의 카테고리를 하나 고르고, 이유를 한 문장으로 설명하라.\n"
        "어느 카테고리와도 관련이 없으면 '분류 불가'를 고른다.\n"
        'JSON 형식으로만 답하라: {"category": "...", "reason": "..."}\n\n'
        f"검색된 예시:\n{examples}\n\n사용자 질문: {question}"
    )
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

if MODE == "openai":
    for question in ["학비를 나눠서 내도 되나요?", "인두기가 고장 났는데 어디에 말해요?", "오늘 날씨 어때?"]:
        emb_pred, _ = classify(question, show=False)
        llm = llm_classify(question)
        print(f"질문: {question}")
        print(f"  임베딩 분류: {emb_pred}")
        print(f"  LLM 분류  : {llm.get('category')}  / 이유: {llm.get('reason')}\n")
else:
    print("API 키가 없어 실습 6을 건너뜁니다. 실습 1~5 결과로 과제를 제출하세요.")

**확인:** 임베딩 분류와 LLM 분류가 다른 질문이 있나요? LLM이 설명한 이유가 검색된 예시와 맞는지 확인하세요.

## 결과 기록표 (노션 결과표에 옮겨 적기)
| 질문 | 1순위 예시(카테고리) | 유사도 | 임베딩 분류 | LLM 분류(선택) | 정답 여부 | 관찰 |
|---|---|---|---|---|---|---|
| 학비를 나눠서 내도 되나요? |  |  |  |  |  |  |
| 랩실 PC 전원이 안 켜져요 |  |  |  |  |  |  |
| 오늘 학식 메뉴 뭐야? |  |  |  |  |  |  |

- 실행 모드(openai/local): ____ / 테스트 정확도: ____ / 최적 threshold: ____
- **7주차 예고:** 오늘 만든 '검색 → 근거 → 생성' 구조에 PDF 문서와 Gradio 화면을 붙여 **RAG 챗봇**을 완성합니다.